# AMICI: Cell-Cell Interaction Inference

This tutorial demonstrates how to use AMICI (Attention-based Multi-scale Interaction for Cell-cell Inference) for spatial transcriptomics analysis.

AMICI uses cross-attention mechanisms to model how neighboring cells influence gene expression through cell-cell interactions.

## Key Features:
- Cross-attention-based interaction modeling
- Cell type-specific interaction patterns
- Latent space representation
- Integration with scvi-tools framework

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

# Import spatialvi
import spatialvi
from spatialvi.external import AMICI

sc.set_figure_params(figsize=(6, 6))
print("spatialvi version:", spatialvi.__version__)

## 1. Load and Prepare Data

In [ ]:
# Load example spatial data
adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
adata.var_names_make_unique()

print(adata)

In [ ]:
# Basic preprocessing
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3")

# For this example, create synthetic cell type labels
# In practice, you would have cell type annotations from deconvolution or segmentation
import pandas as pd

np.random.seed(42)
cell_types = ["B cells", "T cells", "Macrophages", "Dendritic cells", "Fibroblasts"]
adata.obs["cell_type"] = pd.Categorical(
    np.random.choice(cell_types, adata.n_obs)
)

print("\nCell type distribution:")
print(adata.obs["cell_type"].value_counts())

In [ ]:
# Visualize the spatial data
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc.pl.spatial(adata, color="total_counts", spot_size=80, ax=axes[0], show=False)
sc.pl.spatial(adata, color="cell_type", spot_size=80, ax=axes[1], show=False)

plt.tight_layout()
plt.show()

## 2. Compute Spatial Neighbors

AMICI requires precomputed spatial neighbor information.

In [ ]:
from sklearn.neighbors import NearestNeighbors

# Get spatial coordinates
coords = adata.obsm["spatial"]

# Compute k-nearest neighbors
n_neighbors = 10
nn = NearestNeighbors(n_neighbors=n_neighbors + 1)  # +1 to exclude self
nn.fit(coords)
distances, indices = nn.kneighbors(coords)

# Store neighbor information (excluding self)
adata.obsm["nn_index"] = indices[:, 1:].astype(np.int64)
adata.obsm["nn_dist"] = distances[:, 1:].astype(np.float32)

print(f"Computed {n_neighbors} nearest neighbors for each spot")
print(f"Neighbor indices shape: {adata.obsm['nn_index'].shape}")

## 3. Setup and Initialize AMICI Model

In [ ]:
# Setup AnnData for AMICI
AMICI.setup_anndata(
    adata,
    layer=None,  # Use adata.X
    batch_key=None,
    labels_key="cell_type",
    spatial_key="spatial",
    neighbor_index_key="nn_index",
    neighbor_dist_key="nn_dist",
)

print("AnnData setup complete")

In [ ]:
# Initialize AMICI model
model = AMICI(
    adata,
    n_hidden=128,
    n_latent=10,
    n_layers=2,
    n_attention_heads=4,
    dropout_rate=0.1,
    gene_likelihood="nb",  # Negative binomial
    use_cell_type_attention=True,
    interaction_layers=2,
)

print("AMICI model initialized")

## 4. Train the Model

In [ ]:
# Train the model
model.train(
    max_epochs=100,
    batch_size=128,
    early_stopping=True,
    early_stopping_patience=10,
)

print("Training complete!")

## 5. Get Cell-Cell Interaction Scores

In [ ]:
# Get interaction scores
interaction_scores = model.get_interaction_scores()

print(f"Interaction scores shape: {interaction_scores.shape}")

# Store in AnnData
adata.obsm["interaction_scores"] = interaction_scores

In [ ]:
# Get cell type interaction matrix
ct_interactions = model.get_cell_type_interactions(aggregate="mean")

print("Cell type interaction matrix:")
print(ct_interactions.round(3))

In [ ]:
# Visualize cell type interaction matrix as heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    ct_interactions,
    annot=True,
    fmt=".3f",
    cmap="RdBu_r",
    center=0,
    square=True,
)
plt.title("Cell Type Interaction Strengths")
plt.xlabel("Receiver Cell Type")
plt.ylabel("Sender Cell Type")
plt.tight_layout()
plt.show()

## 6. Get Latent Representation

In [ ]:
# Get latent representation
latent = model.get_latent_representation()

print(f"Latent representation shape: {latent.shape}")

# Store in AnnData
adata.obsm["X_amici"] = latent

In [ ]:
# Compute UMAP on latent space
sc.pp.neighbors(adata, use_rep="X_amici")
sc.tl.umap(adata)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc.pl.umap(adata, color="cell_type", ax=axes[0], show=False, title="Cell Types")
sc.pl.spatial(adata, color="cell_type", spot_size=80, ax=axes[1], show=False)

plt.tight_layout()
plt.show()

## 7. Analyze Interaction Patterns

In [ ]:
# Compute mean interaction score per cell
mean_interaction = interaction_scores.mean(axis=1)
adata.obs["mean_interaction"] = mean_interaction

# Visualize spatially
sc.pl.spatial(
    adata,
    color="mean_interaction",
    spot_size=80,
    title="Mean Interaction Score",
    cmap="viridis",
)

In [ ]:
# Compare interaction scores across cell types
plt.figure(figsize=(10, 6))
adata.obs.boxplot(column="mean_interaction", by="cell_type")
plt.ylabel("Mean Interaction Score")
plt.xlabel("Cell Type")
plt.title("Interaction Score Distribution by Cell Type")
plt.suptitle("")  # Remove automatic title
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. Identify High-Interaction Regions

In [ ]:
# Identify spots with high interaction scores
threshold = np.percentile(mean_interaction, 90)
adata.obs["high_interaction"] = (mean_interaction > threshold).astype(str)

print(f"High interaction threshold: {threshold:.4f}")
print(f"Number of high-interaction spots: {(mean_interaction > threshold).sum()}")

# Visualize
sc.pl.spatial(
    adata,
    color="high_interaction",
    spot_size=80,
    title="High Interaction Regions (top 10%)",
)

In [ ]:
# Cell type composition in high-interaction regions
high_int_mask = adata.obs["high_interaction"] == "True"

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# High interaction regions
adata[high_int_mask].obs["cell_type"].value_counts().plot(
    kind="pie", ax=axes[0], autopct="%1.1f%%"
)
axes[0].set_title("High Interaction Regions")
axes[0].set_ylabel("")

# All spots
adata.obs["cell_type"].value_counts().plot(
    kind="pie", ax=axes[1], autopct="%1.1f%%"
)
axes[1].set_title("All Spots")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

## Summary

In this tutorial, we demonstrated:

1. How to prepare spatial data with neighbor information for AMICI
2. How to setup and train the AMICI model
3. How to extract cell-cell interaction scores
4. How to compute cell type interaction matrices
5. How to visualize interaction patterns spatially
6. How to identify high-interaction regions

AMICI provides a powerful framework for understanding cell-cell communication in spatial transcriptomics data using attention-based deep learning.